# 02 User-Centric Fairness Gap

Evaluate accuracy gaps between Niche and Mainstream user groups for six models.

## 1) Setup

In [ ]:
import gc
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

from surprise import Dataset, Reader, KNNBasic, SVD

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = ROOT / 'data'
FIG_DIR = ROOT / 'outputs' / 'figures'
RES_DIR = ROOT / 'outputs' / 'results'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.append(str(ROOT / 'src'))
from utils import load_dataset, ensure_output_dirs
ensure_output_dirs(ROOT)

RANDOM_STATE = 42
TOP_K = 10


## 2) Configuration

In [ ]:
DATASET_NAME = 'ml-100k'
TEST_SIZE = 0.2


## 3) Data Loading and Split

In [ ]:
ratings, item_features = load_dataset(
    dataset_name=DATASET_NAME,
    data_root=DATA_ROOT,
    lastfm_mode='1-5',
    apply_k_core=True,
    min_user_interactions=10,
    min_item_interactions=10,
)
train_df, test_df = train_test_split(ratings, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
all_items = sorted(train_df['item_id'].unique().tolist())
users_test = sorted(test_df['user_id'].unique().tolist())
user_seen = train_df.groupby('user_id')['item_id'].apply(set).to_dict()


## 4) User Mainstreaminess and Quantile Segments

In [ ]:
global_pop = train_df['item_id'].value_counts().to_dict()

user_mainstream = []
for user_id, grp in train_df.groupby('user_id'):
    vals = [global_pop.get(it, 0) for it in grp['item_id'].tolist()]
    score = float(np.mean(vals)) if vals else 0.0
    user_mainstream.append({'user_id': user_id, 'Mainstreaminess_Score': score})

user_mainstream_df = pd.DataFrame(user_mainstream)
q1 = user_mainstream_df['Mainstreaminess_Score'].quantile(0.33)
q2 = user_mainstream_df['Mainstreaminess_Score'].quantile(0.67)


def segment_user(s):
    if s <= q1:
        return 'Niche'
    if s >= q2:
        return 'Mainstream'
    return 'Diverse'

user_mainstream_df['segment'] = user_mainstream_df['Mainstreaminess_Score'].apply(segment_user)
user_segment_map = dict(zip(user_mainstream_df['user_id'], user_mainstream_df['segment']))
user_mainstream_df['segment'].value_counts()


## 5) Utilities

In [ ]:
def popularity_series(df):
    return df['item_id'].value_counts().astype(float)


def gini_coefficient(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return np.nan
    if np.amin(x) < 0:
        x = x - np.amin(x)
    x = x + 1e-9
    x = np.sort(x)
    n = x.size
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * x)) / (n * np.sum(x))


def ndcg_at_k(recommended_items, relevant_items, k=10):
    rel_set = set(relevant_items)
    dcg = 0.0
    for i, it in enumerate(recommended_items[:k], start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(i + 1)
    ideal_hits = min(len(rel_set), k)
    if ideal_hits == 0:
        return 0.0
    idcg = np.sum([1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1)])
    return float(dcg / idcg)


def compute_catalog_coverage(recs, all_items):
    if not recs:
        return 0.0
    rec_items = set([i for items in recs.values() for i in items])
    return len(rec_items) / max(1, len(all_items))


def compute_arp(recs, pop_map):
    vals = []
    for _, items in recs.items():
        vals.extend([pop_map.get(i, 0.0) for i in items])
    return float(np.mean(vals)) if vals else 0.0


def recommendation_frequency(recs):
    freq = {}
    for _, items in recs.items():
        for it in items:
            freq[it] = freq.get(it, 0) + 1
    return pd.Series(freq, dtype=float)

def build_user_seen(train_df):
    return train_df.groupby('user_id')['item_id'].apply(set).to_dict()


def surprise_prepare(train_df):
    reader = Reader(rating_scale=(float(train_df['rating'].min()), float(train_df['rating'].max())))
    data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
    trainset = data.build_full_trainset()
    return trainset


def get_candidate_items(all_items, seen_set):
    if not seen_set:
        return all_items
    return [i for i in all_items if i not in seen_set]


def recommend_with_surprise(algo, users, user_seen, all_items, k=10):
    recs = {}
    for u in tqdm(users, desc='Surprise Top-K'):
        candidates = get_candidate_items(all_items, user_seen.get(u, set()))
        preds = [(it, algo.predict(u, it).est) for it in candidates]
        preds.sort(key=lambda x: x[1], reverse=True)
        recs[u] = [it for it, _ in preds[:k]]
    return recs


def rmse_surprise(algo, test_df):
    preds = [algo.predict(r.user_id, r.item_id).est for r in test_df.itertuples(index=False)]
    return float(np.sqrt(mean_squared_error(test_df['rating'].values, preds)))


def tfidf_item_similarity(item_features):
    tfidf = TfidfVectorizer(min_df=1, max_features=30000)
    mat = tfidf.fit_transform(item_features['metadata_text'])
    sim = cosine_similarity(mat, dense_output=False)
    return sim


def recommend_tfidf(train_df, users, all_items, item_index, sim_matrix, k=10):
    user_hist = train_df.groupby('user_id')['item_id'].apply(list).to_dict()
    recs = {}
    for u in tqdm(users, desc='TF-IDF Top-K'):
        seen = set(user_hist.get(u, []))
        if not seen:
            recs[u] = all_items[:k]
            continue
        seen_idx = [item_index[it] for it in seen if it in item_index]
        if not seen_idx:
            recs[u] = all_items[:k]
            continue
        profile_scores = sim_matrix[seen_idx].mean(axis=0).A1
        scored_items = []
        for it in all_items:
            if it in seen:
                continue
            idx = item_index.get(it)
            if idx is not None:
                scored_items.append((it, profile_scores[idx]))
        scored_items.sort(key=lambda x: x[1], reverse=True)
        recs[u] = [it for it, _ in scored_items[:k]]
    return recs


def train_user_logistic_models(train_df, item_features, user_min_pos=5):
    merged = train_df.merge(item_features, on='item_id', how='left')
    merged['label'] = (merged['rating'] >= merged['rating'].median()).astype(int)
    models = {}
    for user_id, grp in tqdm(merged.groupby('user_id'), desc='LogReg per-user'):
        if grp['label'].nunique() < 2 or len(grp) < user_min_pos:
            continue
        X = grp['metadata_text'].fillna('')
        y = grp['label']
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(min_df=1, max_features=5000)),
            ('clf', LogisticRegression(max_iter=300))
        ])
        pipe.fit(X, y)
        models[user_id] = pipe
    return models


def recommend_user_logreg(models, users, user_seen, item_features, all_items, k=10):
    item_text_map = dict(zip(item_features['item_id'], item_features['metadata_text']))
    recs = {}
    for u in tqdm(users, desc='LogReg Top-K'):
        seen = user_seen.get(u, set())
        candidates = [it for it in all_items if it not in seen]
        m = models.get(u)
        if m is None:
            recs[u] = candidates[:k]
            continue
        X = [item_text_map.get(it, '') for it in candidates]
        if len(X) == 0:
            recs[u] = []
            continue
        probs = m.predict_proba(X)[:, 1]
        order = np.argsort(-probs)[:k]
        recs[u] = [candidates[i] for i in order]
    return recs


def train_rf_regressor(train_df):
    df = train_df.copy()
    ue = LabelEncoder()
    ie = LabelEncoder()
    df['u'] = ue.fit_transform(df['user_id'])
    df['i'] = ie.fit_transform(df['item_id'])
    X = df[['u', 'i']]
    y = df['rating']
    rf = RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X, y)
    return rf, ue, ie


def recommend_rf(rf, ue, ie, users, user_seen, all_items, k=10):
    item_to_enc = {it: idx for idx, it in enumerate(ie.classes_)}
    recs = {}
    for u in tqdm(users, desc='RF Top-K'):
        seen = user_seen.get(u, set())
        candidates = [it for it in all_items if it not in seen and it in item_to_enc]
        if len(candidates) == 0 or u not in set(ue.classes_):
            recs[u] = all_items[:k]
            continue
        u_enc = ue.transform([u])[0]
        X = pd.DataFrame({'u': [u_enc] * len(candidates), 'i': [item_to_enc[it] for it in candidates]})
        preds = rf.predict(X)
        order = np.argsort(-preds)[:k]
        recs[u] = [candidates[i] for i in order]
    return recs


## 6) Train Models + Predictions/Recommendations

In [ ]:
model_preds = {}
model_recs = {}

# k-NN
trainset = surprise_prepare(train_df)
knn = KNNBasic(sim_options={'name': 'cosine', 'user_based': True}, verbose=False)
knn.fit(trainset)
model_preds['k-NN'] = [
    (r.user_id, r.item_id, r.rating, knn.predict(r.user_id, r.item_id).est)
    for r in tqdm(test_df.itertuples(index=False), total=len(test_df), desc='k-NN preds')
]
model_recs['k-NN'] = recommend_with_surprise(knn, users_test, user_seen, all_items, k=TOP_K)
del knn
gc.collect()

# SVD (with Book-Crossing protection)
try:
    svd = SVD(random_state=RANDOM_STATE)
    svd.fit(trainset)
    model_preds['SVD'] = [
        (r.user_id, r.item_id, r.rating, svd.predict(r.user_id, r.item_id).est)
        for r in tqdm(test_df.itertuples(index=False), total=len(test_df), desc='SVD preds')
    ]
    model_recs['SVD'] = recommend_with_surprise(svd, users_test, user_seen, all_items, k=TOP_K)
    del svd
    gc.collect()
except Exception as e:
    print(f'SVD failed (common on sparse Book-Crossing): {e}')
    model_preds['SVD'] = []
    model_recs['SVD'] = {u: all_items[:TOP_K] for u in users_test}

# ALS
try:
    from scipy.sparse import coo_matrix
    from implicit.als import AlternatingLeastSquares

    train_users = sorted(train_df['user_id'].unique().tolist())
    item_to_idx = {it: j for j, it in enumerate(all_items)}

    rows = pd.Categorical(train_df['item_id'], categories=all_items).codes.astype(np.int64)
    cols = pd.Categorical(train_df['user_id'], categories=train_users).codes.astype(np.int64)
    if (rows < 0).any() or (cols < 0).any():
        raise ValueError('ALS indexing failed: found unknown user_id/item_id codes during matrix build.')

    vals = train_df['rating'].astype(float).values
    item_user = coo_matrix((vals, (rows, cols)), shape=(len(all_items), len(train_users))).tocsr()
    user_item = item_user.T.tocsr()

    als = AlternatingLeastSquares(factors=50, iterations=15, regularization=0.01, random_state=RANDOM_STATE)
    als.fit(item_user)

    item_mean = train_df['rating'].mean()
    model_preds['ALS'] = [(r.user_id, r.item_id, r.rating, item_mean) for r in test_df.itertuples(index=False)]

    idx_to_item = {v: k for k, v in item_to_idx.items()}
    user_to_idx = {u: i for i, u in enumerate(train_users)}

    recs = {}
    for u in tqdm(users_test, desc='ALS recs'):
        uid = user_to_idx.get(u)
        if uid is None:
            recs[u] = all_items[:TOP_K]
            continue
        u_items = user_item[uid]
        rec_ids, _ = als.recommend(userid=0, user_items=u_items, N=TOP_K, filter_already_liked_items=True)
        recs[u] = [idx_to_item[i] for i in rec_ids if i in idx_to_item][:TOP_K]

    model_recs['ALS'] = recs
    del als, item_user, user_item
    gc.collect()
except Exception as e:
    print(f'ALS skipped: {e}')
    item_mean = train_df['rating'].mean()
    model_preds['ALS'] = [(r.user_id, r.item_id, r.rating, item_mean) for r in test_df.itertuples(index=False)]
    model_recs['ALS'] = {u: all_items[:TOP_K] for u in users_test}

# TF-IDF
item_features_local = item_features[item_features['item_id'].isin(all_items)].drop_duplicates('item_id').reset_index(drop=True)
item_index = {it: i for i, it in enumerate(item_features_local['item_id'].tolist())}
sim = tfidf_item_similarity(item_features_local)
model_recs['TF-IDF'] = recommend_tfidf(train_df, users_test, all_items, item_index, sim, k=TOP_K)
item_mean = train_df['rating'].mean()
model_preds['TF-IDF'] = [(r.user_id, r.item_id, r.rating, item_mean) for r in test_df.itertuples(index=False)]
del sim
gc.collect()

# LogReg user-specific
logreg_models = train_user_logistic_models(train_df, item_features_local)
model_recs['LogReg'] = recommend_user_logreg(logreg_models, users_test, user_seen, item_features_local, all_items, k=TOP_K)
model_preds['LogReg'] = [(r.user_id, r.item_id, r.rating, item_mean) for r in test_df.itertuples(index=False)]
del logreg_models
gc.collect()

# RandomForest
rf, ue, ie = train_rf_regressor(train_df)
model_recs['RandomForest'] = recommend_rf(rf, ue, ie, users_test, user_seen, all_items, k=TOP_K)
item_to_enc = {it: idx for idx, it in enumerate(ie.classes_)}
model_preds_rf = []
for r in tqdm(test_df.itertuples(index=False), total=len(test_df), desc='RF preds'):
    if (r.user_id in set(ue.classes_)) and (r.item_id in item_to_enc):
        u_enc = ue.transform([r.user_id])[0]
        pred = rf.predict(pd.DataFrame({'u': [u_enc], 'i': [item_to_enc[r.item_id]]}))[0]
    else:
        pred = item_mean
    model_preds_rf.append((r.user_id, r.item_id, r.rating, pred))
model_preds['RandomForest'] = model_preds_rf

del rf, ue, ie
gc.collect()


## 7) Segment-wise RMSE and nDCG@10

In [ ]:
def segment_rmse(pred_rows, user_segment_map):
    df = pd.DataFrame(pred_rows, columns=['user_id', 'item_id', 'true', 'pred'])
    out = {}
    for seg in ['Niche', 'Diverse', 'Mainstream']:
        d = df[df['user_id'].map(user_segment_map) == seg]
        if len(d) == 0:
            out[seg] = np.nan
        else:
            out[seg] = float(np.sqrt(mean_squared_error(d['true'], d['pred'])))
    return out


def segment_ndcg(model_recs, test_df, user_segment_map, k=10):
    rel = test_df.groupby('user_id')['item_id'].apply(set).to_dict()
    rows = {}
    for seg in ['Niche', 'Diverse', 'Mainstream']:
        users_seg = [u for u, s in user_segment_map.items() if s == seg and u in model_recs]
        vals = [ndcg_at_k(model_recs[u], rel.get(u, set()), k=k) for u in users_seg]
        rows[seg] = float(np.mean(vals)) if len(vals) else np.nan
    return rows

results = []
for model_name in model_recs.keys():
    rmses = segment_rmse(model_preds.get(model_name, []), user_segment_map)
    ndcgs = segment_ndcg(model_recs[model_name], test_df, user_segment_map, k=10)
    fairness_gap = rmses.get('Niche', np.nan) - rmses.get('Mainstream', np.nan)
    results.append({
        'model': model_name,
        'rmse_niche': rmses.get('Niche', np.nan),
        'rmse_diverse': rmses.get('Diverse', np.nan),
        'rmse_mainstream': rmses.get('Mainstream', np.nan),
        'ndcg_niche': ndcgs.get('Niche', np.nan),
        'ndcg_diverse': ndcgs.get('Diverse', np.nan),
        'ndcg_mainstream': ndcgs.get('Mainstream', np.nan),
        'delta_accuracy_rmse_niche_minus_mainstream': fairness_gap,
    })

fairness_df = pd.DataFrame(results).sort_values('model')
fairness_df


## 8) Visualization: RMSE by User Segment

In [ ]:
plot_df = fairness_df.melt(
    id_vars=['model'],
    value_vars=['rmse_niche', 'rmse_diverse', 'rmse_mainstream'],
    var_name='segment',
    value_name='rmse',
)
plot_df['segment'] = plot_df['segment'].str.replace('rmse_', '', regex=False).str.title()

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x='model', y='rmse', hue='segment')
plt.title(f'RMSE by User Segment ({DATASET_NAME})')
plt.xticks(rotation=20)
plt.tight_layout()
out_path = FIG_DIR / f'user_centric_rmse_segment_{DATASET_NAME}.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved figure: {out_path}')


## 9) Save Outputs

In [ ]:
out_csv = RES_DIR / f'user_centric_metrics_{DATASET_NAME}.csv'
fairness_df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')
